# Phase 1: Simulating the Data
## Heavy Equipment Supply Chain

### Features:
- order_date:                  When the purchase order was placed.
- po_id:                       Unique purchase order identifier.
- supplier_id:                 Unique supplier identifier.
- supplier_region:             
- part_id:                     Unique part identifier.
- part_type:                   Type of part.
- category:                    Groups spend into procurement-relevant buckets.
- unit_of_measure             
- quantity                     
- unit_cost_usd              
- total_cost_usd               
- promised_lead_time_days:      Agreed lead time at order. Used to compare to actual.
- actual_lead_time_days:        Actual delivery lead time. Allows calculation of schedule adherence.     
- on_time_flag:                 Simplified metric for OTIF (On-Time, In-Full) KPI.
- defect_flag:                  Supplier quality at receipt. Critical for defect rate. 
- defect_reason:                Insight into quality failure modes.


In [1]:
# Import required libraries
import pandas as pd
import random 
import uuid
from typing import Dict, List, Tuple, Optional, TypedDict
from datetime import datetime, timedelta

## 1. Create catalog
### a. Create categories
### b. Create types
### c. Create units map
### d. Create prices
### e. Create part ids

In [2]:
# ----Create the principal dictionaries necessary for catalog construction----

# Create main categories
component_categories = ["Steel", "Hydraulics", "Engines", "Electronics", "Undercarriage", "Filters", "Fasteners", "Cabs & Controls","Attachments", "Lubricants & Fluids"]

# Create components per category
component_types = {
    "Steel": ["Steel Plate", "Steel Beam", "Forged Steel Pin", "Steel Casting"],
    "Hydraulics": ["Hydraulic Cylinder", "Hydraulic Hose", "Hydraulic Pump", "Control Valve", "Seal Kit"],
    "Engines": ["Diesel Engine", "Engine Block", "Piston", "Fuel Injector", "Turbocharger"],
    "Electronics": ["ECM (Engine Control Module)", "Wiring Harness", "Pressure Sensor", "Temperature Sensor", "GPS Module"],
    "Undercarriage": ["Track Link", "Track Roller", "Idler", "Sprocket", "Wheel Rim"],
    "Filters": ["Engine Oil Filter", "Fuel Filter", "Hydraulic Filter", "Air Filter", "Cabin Filter"],
    "Fasteners": ["Hex Bolt", "Nut", "Washer", "Stud"],
    "Cabs & Controls": ["Operator Seat", "Joystick", "Instrument Panel", "Cab Glass"],
    "Attachments": ["Bucket", "Blade", "Ripper", "Fork Assembly"],
    "Lubricants & Fluids": ["Engine Oil", "Hydraulic Fluid", "Coolant", "Transmission Oil"]
    }

# Define units of measure
measure_units = ["ea", "pk", "lb", "gal", "ft"]

# Map components per units of measure
component_unit_map = {
        "Steel Plate": "lb",
    "Steel Beam": "ft",
    "Forged Steel Pin": "ea",
    "Steel Casting": "lb",
    "Hydraulic Cylinder": "ea",
    "Hydraulic Hose": "ft",
    "Hydraulic Pump": "ea",
    "Control Valve": "ea",
    "Seal Kit": "pk",
    "Diesel Engine": "ea",
    "Engine Block": "ea",
    "Piston": "ea",
    "Fuel Injector": "ea",
    "Turbocharger": "ea",
    "ECM (Engine Control Module)": "ea",
    "Wiring Harness": "ft",
    "Pressure Sensor": "ea",
    "Temperature Sensor": "ea",
    "GPS Module": "ea",
    "Track Link": "ea",
    "Track Roller": "ea",
    "Idler": "ea",
    "Sprocket": "ea",
    "Wheel Rim": "ea",
    "Engine Oil Filter": "ea",
    "Fuel Filter": "ea",
    "Hydraulic Filter": "ea",
    "Air Filter": "ea",
    "Cabin Filter": "ea",
    "Hex Bolt": "ea",
    "Nut": "ea",
    "Washer": "ea",
    "Stud": "ea",
    "Operator Seat": "ea",
    "Joystick": "ea",
    "Instrument Panel": "ea",
    "Cab Glass": "ea",
    "Bucket": "ea",
    "Blade": "ea",
    "Ripper": "ea",
    "Fork Assembly": "ea",
    "Engine Oil": "gal",
    "Hydraulic Fluid": "gal",
    "Coolant": "gal",
    "Transmission Oil": "gal"
}

# Research actual market and create price ranges per component
detailed_price_ranges = {
    "Steel": {
        # Per-lb or per-ft where noted
        "Steel Plate": {
            "A36/Mild (per lb)": (0.60, 1.00),
            "High-strength/QT (per lb)": (1.20, 2.50)
        },
        "Steel Beam": {
            "Common sections (per ft)": (6.00, 30.00)
        },
        "Forged Steel Pin": {
            "Standard (ea)": (20.00, 150.00)
        },
        "Steel Casting": {
            "Carbon steel (per lb)": (1.50, 4.00)
        }
    },

    "Hydraulics": {
        "Hydraulic Cylinder": {
            "2 to 3.5 in bore, 24 in stroke (ea)": (175.00, 450.00),
            "Large-bore / HD (ea)": (600.00, 2000.00)
        },
        "Hydraulic Hose": {
            "3/8 in 2-wire (per ft)": (2.50, 6.00)
        },
        "Hydraulic Pump": {
            "Gear pump (ea)": (100.00, 600.00),
            "Piston/axial (ea)": (800.00, 2500.00)
        },
        "Control Valve": {
            "1 to 4 spool, 3000 psi (ea)": (150.00, 600.00)
        },
        "Seal Kit": {
            "Cylinder seal kit (pk)": (10.00, 100.00)
        }
    },

    "Engines": {
        "Diesel Engine": {
            "4 to 6 cyl industrial (ea)": (15000.00, 40000.00)
        },
        "Engine Block": {
            "Bare/cast (ea)": (1500.00, 6000.00)
        },
        "Piston": {
            "Heavy diesel (ea)": (40.00, 150.00)
        },
        "Fuel Injector": {
            "HEUI/common-rail (ea)": (350.00, 900.00)
        },
        "Turbocharger": {
            "BorgWarner S300 class (ea)": (900.00, 2500.00)
        }
    },

    "Electronics": {
        "ECM (Engine Control Module)": {
            "Reman/OEM (ea)": (800.00, 3000.00)
        },
        "Wiring Harness": {
            "Primary wire (per ft)": (0.20, 0.90)
        },
        "Pressure Sensor": {
            "Industrial 0 to 3000 psi, 4 to 20 mA (ea)": (80.00, 250.00)
        },
        "Temperature Sensor": {
            "Automotive NTC (ea)": (10.00, 40.00)
        },
        "GPS Module": {
            "u-blox class GNSS (ea)": (25.00, 160.00)
        }
    },

    "Undercarriage": {
        "Track Link": {
            "Link assembly per side (ea)": (2000.00, 5500.00)
        },
        "Track Roller": {
            "Bottom (ea)": (150.00, 400.00),
            "Top/Carrier (ea)": (100.00, 250.00)
        },
        "Idler": {
            "Front idler (ea)": (300.00, 1100.00)
        },
        "Sprocket": {
            "Drive sprocket (ea)": (150.00, 600.00)
        },
        "Wheel Rim": {
            "Heavy equipment (ea)": (150.00, 600.00)
        }
    },

    "Filters": {
        "Engine Oil Filter": { "Standard (ea)": (8.00, 25.00) },
        "Fuel Filter": { "Diesel (ea)": (20.00, 80.00) },
        "Hydraulic Filter": { "Spin-on/element (ea)": (20.00, 100.00) },
        "Air Filter": { "Primary/outer (ea)": (20.00, 70.00) },
        "Cabin Filter": { "HVAC (ea)": (10.00, 30.00) }
    },

    "Fasteners": {
        "Hex Bolt": { "Grade 5/8 (ea)": (0.10, 6.00) },
        "Nut": { "Steel (ea)": (0.05, 2.00) },
        "Washer": { "Steel (ea)": (0.02, 0.50) },
        "Stud": { "Assorted (ea)": (1.00, 10.00) }
    },

    "Cabs & Controls": {
        "Operator Seat": { "Mech/air suspension (ea)": (500.00, 1900.00) },
        "Joystick": { "Hyd/elec (ea)": (100.00, 600.00) },
        "Instrument Panel": { "Cluster/IPC (ea)": (300.00, 1500.00) },
        "Cab Glass": { "OEM-equivalent (ea)": (140.00, 900.00) }
    },

    "Attachments": {
        "Bucket": { "Mini to midsize (ea)": (2000.00, 7000.00) },
        "Blade": { "Dozer/grading (ea)": (5000.00, 9000.00) },
        "Ripper": { "Tooth / shank (ea)": (500.00, 4000.00) },
        "Fork Assembly": { "48 in, 4k lb (ea)": (900.00, 1500.00) }
    },

    "Lubricants & Fluids": {
        # Per gallon
        "Engine Oil": { "15W-40 (per gal)": (15.00, 30.00) },
        "Hydraulic Fluid": { "ISO 46 (per gal)": (12.00, 25.00) },
        "Coolant": { "ELC 50/50 (per gal)": (12.00, 25.00) },
        "Transmission Oil": { "ATF / powershift (per gal)": (15.00, 35.00) }
    }
}

In [3]:
# Generate id per part
def generate_part_id():
    """ Generate a unique part ID """ 
    return f"PART-{uuid.uuid4().hex[:8]}"

# Generate component price 
def pick_price(
    category: str,
    part_type: str,
    detailed_map: Dict[str, Dict[str, Dict[str, Tuple[float, float]]]],
    rng: random.Random,
) -> Tuple[Optional[float], Optional[str]]:
    """
    Returns (price, variant_label). If no detailed price exists, returns (None, None).
    """
    subtree = detailed_map.get(category, {}).get(part_type)
    if isinstance(subtree, dict) and subtree:
        variant, (lo, hi) = rng.choice(list(subtree.items()))
        return round(rng.uniform(lo, hi), 2), variant
    return None, None

# -----------------------------------------------
# Create catalog
def create_catalog(
    types_map: Dict[str, List[str]],
    unit_map: Dict[str, str],
    detailed_map: Dict[str, Dict[str, Dict[str, Tuple[float, float]]]],
    seed: Optional[int] = None,
    on_missing: str = "null",  # "null" leave price None,
) -> List[Dict[str, object]]:
    """
    Builds the catalog without any synthetic fallback pricing.

    on_missing:
      - "null": unit_cost_usd stays None if not found (default).
      - "error": raise ValueError if a part has no priced variant.
    """
    rng = random.Random(seed)
    catalog: List[Dict[str, object]] = []

    for category, types in types_map.items():
        for part_type in types:
            unit = unit_map.get(part_type, "ea")
            price, variant = pick_price(category, part_type, detailed_map, rng)

            if price is None and on_missing == "error":
                raise ValueError(f"No price data for category='{category}', part='{part_type}'")

            catalog.append(
                {
                    "part_id": generate_part_id(),
                    "category": category,
                    "type": part_type,
                    "unit_of_measure": unit,
                    "unit_cost_usd": price,  # None when unknown
                    "pricing_variant": variant,  # None when unknown
                }
            )
    return catalog

## 2. Generate suppliers

In [4]:
# ---- Create the principal dictionaries needed for catalog construction ----

# Asign countries per category
category_country_map = {
    "Steel": ["China","India","Japan","South Korea","USA","Brazil","Germany","Turkey"],
    "Hydraulics": ["USA","Germany","Japan","China","Italy"],
    "Engines": ["USA","Japan","Germany","China","India"],
    "Electronics": ["China","Taiwan","South Korea","Japan","USA"],
    "Undercarriage": ["USA","China","Mexico","India","Brazil"],
    "Filters": ["USA","China","Germany","Japan"],
    "Fasteners": ["China","India","Taiwan","Italy"],
    "Cabs & Controls": ["USA","Japan","Mexico","China"],
    "Attachments": ["USA","China","India","Brazil"],
    "Lubricants & Fluids": ["USA","Germany","Netherlands","China"]    
}

# Define Regions and countries
supplier_region_country = {
    "North America": ["USA", "Mexico"],
    "Asia": ["China", "India", "Japan", "South Korea", "Taiwan", "Thailand"], 
    "Europe" : ["United Kingdom", "Germany","Netherlands", "Italy", "France"],
    "South America": ["Brazil"]
}

# Create profile per supplier
supplier_profiles = {
    "Supplier A": {"categories": ["Steel", "Electronics"], "on_time_rate": 0.98, "quality_score": 0.95, "lead_time": 14},
    "Supplier B": {"categories": ["Hydraulics"], "on_time_rate": 0.95, "quality_score": 0.93, "lead_time": 12},
    "Supplier C": {"categories": ["Engines"], "on_time_rate": 0.92, "quality_score": 0.96, "lead_time": 35},
    "Supplier D": {"categories": ["Undercarriage"], "on_time_rate": 0.94, "quality_score": 0.92, "lead_time": 18},
    "Supplier E": {"categories": ["Filters", "Fasteners"], "on_time_rate": 0.99, "quality_score": 0.94, "lead_time": 7},
    "Supplier F": {"categories": ["Attachments"], "on_time_rate": 0.93, "quality_score": 0.91, "lead_time": 21},
    "Supplier G": {"categories": ["Lubricants & Fluids"], "on_time_rate": 0.97, "quality_score": 0.93, "lead_time": 5},
    "Supplier H": {"categories": ["Cabs & Controls"], "on_time_rate": 0.95, "quality_score": 0.92, "lead_time": 16},
    "Supplier I": {"categories": ["Hydraulics", "Electronics"], "on_time_rate": 0.94, "quality_score": 0.94, "lead_time": 14},
    "Supplier J": {"categories": ["Steel", "Undercarriage"], "on_time_rate": 0.93, "quality_score": 0.91, "lead_time": 20},
    "Supplier K": {"categories": ["Engines", "Attachments"], "on_time_rate": 0.92, "quality_score": 0.93, "lead_time": 30},
    "Supplier L": {"categories": ["Filters", "Lubricants & Fluids"], "on_time_rate": 0.98, "quality_score": 0.95, "lead_time": 6}
}
# Type hints
class SupplierProfile(TypedDict,total=False):
    categories: List[str]
    on_time_rate: float
    lead_time: int

# Find region
def find_region(country, supplier_region_country):
    for region, countries in supplier_region_country.items():
        if country in countries:
            return region
    return None


# Generate suppliers
def generate_suppliers(    
    supplier_profiles: SupplierProfile,
    category_country_map: Dict[str, List[str]],
    supplier_region_country: Dict[str, List[str]],
    seed: Optional[int] = None,
    region_price_bias: Optional[Dict[str, float]] = None,
    multiplier_jitter: tuple[float, float] = (0.98, 1.04),      
) -> list[dict]:
    """
    Build supplier records with deterministic (seeded) multipliers.
    Returns a list of dicts with:
    - supplier_id:     "SU-001", "SU-002", ...
    - name:            Supplier display name
    - region:          Derived from country via supplier_region_country
    - country:         Picked from category_country_map[category]
    - category:        One of the supplier's supported categories
    - price_multiplier:region_bias * jitter  (rounded to 3 decimals)
    - on_time_rate:    From supplier_profiles (rounded to 3 decimals)
    - quality_score:   From supplier_profiles (rounded to 3 decimals)
    - base_lead_time_days: From supplier_profiles (int)

Notes:
    - If a category has no mapped countries, we leave country=None and region=None.
    - Region bias is intentionally narrow and configurable.
        - Set all biases to 1.0 for purely neutral multipliers.
    """
    rng = random.Random(seed)
    region_price_bias = region_price_bias or {
        "North America": 1.00,
        "Europe": 1.03,
        "Asia": 0.99,
        "South America": 1.02,
    }

    # Invert supplier_region_country for quick lookup
    country_to_region = {}
    for region, countries in supplier_region_country.items():
        for c in countries:
            country_to_region[c] = region

    records: List[Dict[str, object]] = []

    # Asign each supplier a stable id in insertion order
    for i, (sup_name, prof) in enumerate(supplier_profiles.items(), start=1):
        categories: List[str] = prof.get("categories", [])
        on_time = float(prof.get("on_time_rate", 0.95))
        quality = float(prof.get("quality_score", 0.95))
        base_lead = int(prof.get("lead_time", 14))

        for cat in categories:
            countries = category_country_map.get(cat, []) 
            country = rng.choice(countries) if countries else None
            region = country_to_region.get(country)  if country else None

            # Compose multiplier
            bias = region_price_bias.get(region, 1.0)
            lo, hi = multiplier_jitter
            jitter = rng.uniform(lo, hi)
            m = round(bias * jitter, 3)

            records.append({
                "supplier_id": f"SU-{str(i).zfill(3)}",
                "name": sup_name,
                "region": region,
                "country": country,
                "category": cat,
                "price_multiplier": m,
                "on_time_rate": round(on_time, 3),
                "quality_score": round(quality, 3),
                "base_lead_time_days": base_lead,
            })

    return records

## 3. Sampling

In [5]:
# Define defect rates by categories
defect_rate_by_category = {
    "Engines": 0.02,
    "Attachments": 0.04,
    "Steel": 0.03,
    "Hydraulics": 0.03,
    "Electronics": 0.05,
    "Undercarriage": 0.03,
    "Filters": 0.01,
    "Fasteners": 0.005,
    "Lubricants & Fluids": 0.002,
    "Cabs & Controls": 0.02
}
# Define lead time distributions (min_days, max_days)
lead_time_ranges = {
    "Engines": (21, 90),
    "Attachments": (14, 60),
    "Steel": (7, 28),
    "Hydraulics": (7, 30),
    "Electronics": (7, 45),
    "Undercarriage": (14, 45),
    "Filters": (2, 10),
    "Fasteners": (1, 7),
    "Lubricants & Fluids": (1, 7),
    "Cabs & Controls": (14, 45)
}

# Pricing & quantity helpers 
def sample_unit_price(part, supplier, noise_range=(1.0, 1.0)):
    """
    Returns the offered unit price or None if the base is None.
    noise_range=(1.0,1.0) disables extra noise; set e.g. (0.99,1.02) to allow ±1–2%.
    """
    base = part.get("unit_cost_usd")
    if base is None:
        return None
    lo, hi = noise_range
    noise = random.uniform(lo, hi)
    return round(base * supplier["price_multiplier"] * noise, 2)

def sample_quantity(part):
    cat = part["category"]
    if cat == "Engines": return random.randint(1, 3)
    if cat == "Attachments": return random.randint(1, 4)
    if cat == "Filters": return random.randint(5, 200)
    if cat == "Fasteners": return random.randint(50, 1000)
    if cat == "Steel": return random.randint(50, 5000)      # lb/ft, etc.
    if cat == "Hydraulics": return random.randint(1, 20)
    if cat == "Lubricants & Fluids": return random.randint(1, 200)
    if cat == "Electronics": return random.randint(1, 30)
    return random.randint(1, 50)

def sample_promised_lead_days(category, supplier_base_lead, lead_time_ranges):
    """
    Combine supplier base with category variability.
    Simple rule: promised = max(supplier_base_lead, category_draw)
    """
    lo, hi = lead_time_ranges.get(category, (7, 30))
    category_draw = random.randint(lo, hi)
    return max(supplier_base_lead, category_draw)

# Generate orders 
def generate_orders(parts_catalog, suppliers, start_date, end_date,
                    orders_per_day=10, seed=None, include_end=True,
                    price_noise_range=(1.0, 1.0)): 
    """
    Returns a pandas DataFrame of orders.
    - Skips parts with unit_cost_usd is None (strict-null).
    - Promised lead time = max(supplier.base_lead_time_days, category range draw).
    - Actual lead time ~ N(promised, 20%*promised), clipped to >=1.
    """
    if seed is not None:
        random.seed(seed)

    # validations
    required_supplier_keys = {"supplier_id", "region", "country", "category",
                              "base_lead_time_days", "on_time_rate"}
    for s in suppliers:
        missing = required_supplier_keys - s.keys()
        if missing:
            raise ValueError(f"Supplier missing keys {missing}: {s}")

    required_part_keys = {"part_id", "type", "category", "unit_of_measure", "unit_cost_usd"}
    for p in parts_catalog:
        missing = required_part_keys - p.keys()
        if missing:
            raise ValueError(f"Part missing keys {missing}: {p}")

    if orders_per_day < 0:
        raise ValueError("orders_per_day must be >= 0")

    day_span = (end_date - start_date).days + (1 if include_end else 0)
    if day_span <= 0:
        return pd.DataFrame(columns=[
            "order_date","po_id","supplier_id","supplier_region","supplier_country",
            "part_id","part_type","category","unit_of_measure","quantity","unit_cost_usd",
            "total_cost_usd","promised_lead_time_days","actual_lead_time_days",
            "on_time_flag","defect_flag","defect_reason"
        ])

    orders = []
    for day_offset in range(day_span):
        order_date = (start_date + timedelta(days=day_offset)).date()
        for _ in range(orders_per_day):
            supplier = random.choice(suppliers)
            supplier_category = supplier["category"]
            # Only consider priced parts in this category
            candidates = [p for p in parts_catalog
                          if p["category"] == supplier_category and p["unit_cost_usd"] is not None]
            if not candidates:
                # fallback: any priced part (ensures strict-null)
                candidates = [p for p in parts_catalog if p["unit_cost_usd"] is not None]
                if not candidates:
                    # nothing priced at all; stop early
                    continue
            part = random.choice(candidates)

            quantity = sample_quantity(part)
            unit_cost = sample_unit_price(part, supplier, noise_range=price_noise_range)
            if unit_cost is None:
                # Shouldn’t happen due to filtering, but guard anyway
                continue
            total_cost = round(quantity * unit_cost, 2)

            promised = sample_promised_lead_days(
                part["category"],
                supplier["base_lead_time_days"],
                lead_time_ranges
            )

            # The actual lead time is gaussian around promised with sd = 20% of promised
            actual = max(1, int(round(random.gauss(promised, max(1, promised * 0.2)))))

            # First compute whether they meet promised naturally, then adjust:
            meets_promise = actual <= promised
            # If they missed but supplier has high on_time_rate, occasionally pull it back in:
            if not meets_promise and random.random() < supplier["on_time_rate"]:
                actual = promised  # nudge to on-time
                meets_promise = True

            on_time = int(meets_promise)

            defect_prob = defect_rate_by_category.get(part["category"], 0.01)
            defect_flag = int(random.random() < defect_prob)
            defect_reason = None
            if defect_flag:
                defect_reason = random.choice([
                    "Incorrect Specs", "Damaged in Transit", "Nonconformance", "Packaging Issue"
                ])
            else:
                defect_reason = "No Defect"

            po_id = f"PO-{uuid.uuid4().hex[:8]}"

            orders.append({
                "order_date": order_date.isoformat(),
                "po_id": po_id,
                "supplier_id": supplier["supplier_id"],
                "supplier_region": supplier["region"],
                "supplier_country": supplier["country"],
                "part_id": part["part_id"],
                "part_type": part["type"],
                "category": part["category"],
                "unit_of_measure": part["unit_of_measure"],
                "quantity": quantity,
                "unit_cost_usd": unit_cost,
                "total_cost_usd": total_cost,
                "promised_lead_time_days": promised,
                "actual_lead_time_days": actual,
                "on_time_flag": on_time,
                "defect_flag": defect_flag,
                "defect_reason": defect_reason
            })

    return pd.DataFrame(orders)

In [6]:
# Call functions and create DataFrame
parts = create_catalog(component_types, component_unit_map, detailed_price_ranges, seed=42)

sups = generate_suppliers(
    supplier_profiles=supplier_profiles,
    category_country_map=category_country_map,
    supplier_region_country=supplier_region_country,
    seed=42
)

df = generate_orders(parts, sups, datetime(2023,1,1), datetime(2025,9,1),
                     orders_per_day=10, seed=3, price_noise_range=(1.0, 1.0))


In [7]:
df.to_csv('/Users/AndreaLopera/Desktop/Data Science Portfolio/Procurement-Intelligence-Dashboard-main/data/synthetic_procurement_orders.csv', index=False)